# 16 — Modelo de partidos con features pre-partido: Liga MX

Este notebook contesta la pregunta pendiente: **¿sirve el perfil histórico de equipo, metido
como variable de entrada, para predecir de verdad un partido antes de que se juegue?**

## Por qué este modelo es distinto al del notebook 11

En el notebook 11 (Premier League), el modelo de ganador usaba `possession_diff` y
`shots_on_target_diff` — estadísticas del **mismo partido** que se intentaba predecir. Eso lo
convierte en un modelo **explicativo** (qué se asocia con el resultado, mirando el partido ya
jugado), no uno que sirva para predecir antes de que empiece.

Aquí usamos exclusivamente variables que **sí se conocen antes del pitazo inicial**:

- El **perfil histórico** de cada equipo (goles, corners y tarjetas que genera/recibe cuando
  juega de local o de visita), del notebook 15.
- La **severidad histórica del árbitro** asignado (`arbitro_dureza`), del notebook 14.

## El detalle que evita hacer trampa sin darse cuenta

No basta con usar "variables históricas" en general — hay que tener cuidado de que, para CADA
partido, el perfil que se usa como feature de ese partido no incluya información del propio
partido ni de partidos futuros a él. Por eso, para cada partido, el perfil de equipo y la dureza
del árbitro se recalculan usando **solo las temporadas estrictamente anteriores** a la temporada
de ese partido — la misma idea de ventana "expandible" que ya probamos en la sección 8 del
notebook 15 (ahí vimos que expandible funciona tan bien como cualquier ventana móvil corta, así
que la usamos aquí por ser la más simple).

Esto tiene una consecuencia importante: los partidos de la **primera temporada** (2020-21) no
tienen ninguna temporada anterior de la cual sacar un perfil, así que **se excluyen del modelado**
— no hay forma honesta de darles un feature pre-partido.

## 1. Carga de datos y variable objetivo (ganador)

Igual que en el notebook 15, partimos del dataset `v3`. Aquí además definimos el **ganador** de
cada partido (Local / Empate / Visitante) a partir de `home_goals` y `away_goals` — la variable
que el "Modelo A" de este notebook va a intentar predecir.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option("display.max_columns", 40)

df = pd.read_csv("../data/processed/matches_ligamx_2020_2025_v3_arbitraje.csv", dtype=str)

# Corrección de calidad de datos descubierta en el notebook 14: 6 árbitros
# quedaron duplicados en la fuente porque FBref cambió el formato de sus
# nombres a partir de 2023-24 (agregó acentos y/o segundo nombre). Se aplica
# aquí, justo al cargar los datos, para que TODO lo que sigue en este
# notebook (incluyendo el cálculo de dureza por árbitro más abajo) trabaje
# con la identidad correcta desde el inicio, en vez de arrastrar el bug.
mapa_nombres_arbitro = {
    "Victor Caceres": "Víctor Cáceres", "Oscar Mejia": "Oscar Mejía",
    "Marco Ortíz": "Marco Antonio Ortíz", "Luis Santander": "Luis Enrique Santander",
    "Erick Miranda": "Erick Yair Miranda", "Guillermo Pacheco": "Guillermo Pacheco Larios",
    "Jesus Lopez": "Jesús López", "Ismael Lopez": "Ismael López",
}
df["referee"] = df["referee"].replace(mapa_nombres_arbitro)

columnas_numericas = [
    "home_shots_total", "away_shots_total",
    "home_yellow", "home_red", "home_total_cards",
    "away_yellow", "away_red", "away_total_cards", "total_game_cards",
    "home_fouls", "away_fouls", "home_corners", "away_corners",
]
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Mismo regex que en los notebooks 13-15: separa el marcador en tiempo
# reglamentario, ignorando penales entre paréntesis en partidos de liguilla.
patron_score = r"(?:\(\d+\)\s*)?(\d+)\s*[\u2013-]\s*(\d+)(?:\s*\(\d+\))?"
goles = df["score"].str.extract(patron_score)
df["home_goals"] = pd.to_numeric(goles[0], errors="coerce")
df["away_goals"] = pd.to_numeric(goles[1], errors="coerce")

# Variable objetivo del Modelo A: quién ganó. Empate incluye marcador
# empatado en tiempo reglamentario, aunque la liguilla se haya definido
# después por penales — nos interesa el resultado "en cancha", no el
# avance de ronda.
condiciones = [df["home_goals"] > df["away_goals"], df["home_goals"] < df["away_goals"]]
df["ganador"] = np.select(condiciones, ["Local", "Visitante"], default="Empate")

print(df["ganador"].value_counts())
print(f"\nProporción Local: {(df['ganador']=='Local').mean():.1%} (será nuestro baseline ingenuo)")

ganador
Local        953
Visitante    598
Empate       551
Name: count, dtype: int64

Proporción Local: 45.3% (será nuestro baseline ingenuo)


## 2. Perfiles y dureza de árbitro "hasta antes de" cada temporada

Reutilizamos la lógica de `a_formato_largo` (notebook 15) y construimos, para cada temporada, el
perfil de equipo y la dureza de árbitro usando SOLO las temporadas anteriores — igual que la
ventana "expandible" de la sección 8 del notebook 15, aquí aplicada de una vez para las 4
temporadas que sí tienen historia previa (2021-22 en adelante).

In [2]:
def a_formato_largo(partidos):
    """Idéntica a la función del notebook 15: convierte un DataFrame a nivel
    partido en uno a nivel equipo-partido (una fila por equipo por partido),
    con `es_local` indicando la perspectiva. Se repite aquí (en vez de
    importarla) porque los notebooks de este proyecto están pensados para
    poder re-ejecutarse de forma independiente, sin depender de que otro
    notebook se haya corrido antes en la misma sesión.
    """
    local = pd.DataFrame({
        "season": partidos["season"], "team": partidos["home_team"], "rival": partidos["away_team"],
        "es_local": 1, "goals_for": partidos["home_goals"], "goals_against": partidos["away_goals"],
        "corners_for": partidos["home_corners"], "cards_for": partidos["home_total_cards"],
    })
    visitante = pd.DataFrame({
        "season": partidos["season"], "team": partidos["away_team"], "rival": partidos["home_team"],
        "es_local": 0, "goals_for": partidos["away_goals"], "goals_against": partidos["home_goals"],
        "corners_for": partidos["away_corners"], "cards_for": partidos["away_total_cards"],
    })
    return pd.concat([local, visitante], ignore_index=True)


temporadas_orden = sorted(df["season"].unique())
metricas_perfil = ["goals_for", "goals_against", "corners_for", "cards_for"]


def perfil_hasta(temporada_objetivo):
    """Perfil equipo-contexto (Local/Visita) calculado SOLO con las
    temporadas estrictamente anteriores a `temporada_objetivo`.

    Devuelve None si `temporada_objetivo` es la primera temporada del
    dataset (no hay historia previa posible) — quien llame a esta función
    debe manejar ese caso descartando esos partidos del modelado.
    """
    i = temporadas_orden.index(temporada_objetivo)
    previas = temporadas_orden[:i]
    if not previas:
        return None
    subset = df[df["season"].isin(previas)]
    largo = a_formato_largo(subset)
    return largo.groupby(["team", "es_local"])[metricas_perfil].mean()


def dureza_arbitro_hasta(temporada_objetivo):
    """Análogo a `perfil_hasta`, pero para la severidad histórica de cada
    árbitro (promedio de `total_game_cards` en sus partidos de temporadas
    anteriores). No hace falta corregir nombres duplicados aquí porque ya se
    consolidaron una sola vez sobre `df` completo justo después de cargarlo
    (celda anterior) — todo lo que sigue ya trabaja con la identidad
    correcta de cada árbitro.
    """
    i = temporadas_orden.index(temporada_objetivo)
    previas = temporadas_orden[:i]
    if not previas:
        return None
    subset = df[df["season"].isin(previas)]
    return subset.groupby("referee")["total_game_cards"].mean()


print("Temporadas con perfil disponible (todas menos la primera):", temporadas_orden[1:])

Temporadas con perfil disponible (todas menos la primera): ['2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026', '2026-2027']


## 3. Tabla de features por partido

Para cada partido (excepto los de la primera temporada, que no tienen historia previa),
armamos una fila con:

- El perfil **del equipo local, en su contexto de local** (`local_goles_favor`,
  `local_goles_contra`, `local_corners_favor`, `local_tarjetas_favor`) — cómo se comporta ESE
  equipo específicamente cuando juega en casa, no un promedio genérico.
- El perfil **del equipo visitante, en su contexto de visita** (mismas 4 métricas).
- La **dureza histórica del árbitro** asignado a ese partido.

Todo calculado usando `perfil_hasta`/`dureza_arbitro_hasta` con la temporada de ESE partido como
`temporada_objetivo` — es decir, cada partido usa el conocimiento acumulado hasta el final de la
temporada anterior a la suya, nunca información de su propia temporada ni de temporadas futuras.

In [3]:
def construir_features(df_partidos):
    """Arma la tabla de features pre-partido para un conjunto de partidos.

    Se apoya en un pequeño cache (`cache_perfiles`, `cache_dureza`) para no
    recalcular el mismo perfil/dureza una y otra vez para cada partido de la
    misma temporada — perfil_hasta/dureza_arbitro_hasta ya son algo costosas
    porque reconstruyen el formato largo desde cero cada vez que se llaman.
    """
    cache_perfiles = {}
    cache_dureza = {}
    filas = []

    for _, partido in df_partidos.iterrows():
        temporada = partido["season"]
        if temporada not in cache_perfiles:
            cache_perfiles[temporada] = perfil_hasta(temporada)
            cache_dureza[temporada] = dureza_arbitro_hasta(temporada)
        perfil = cache_perfiles[temporada]
        dureza = cache_dureza[temporada]

        if perfil is None or dureza is None:
            continue  # primera temporada del dataset, sin historia previa: se descarta

        # OJO: `perfil` es un DataFrame (viene de agrupar por VARIAS columnas
        # de métrica a la vez, no una sola), así que `.get((equipo, es_local))`
        # NO sirve para buscar una fila por su índice — `.get()` en un
        # DataFrame busca una COLUMNA con ese nombre, y como no existe
        # ninguna columna llamada así, siempre devuelve None silenciosamente
        # (sin error) y se descartarían TODOS los partidos sin darse cuenta.
        # La forma correcta de buscar una fila por índice compuesto es
        # `.loc[...]`, envuelto en try/except porque sí lanza KeyError si el
        # equipo no tiene historia previa en ese contexto.
        try:
            local_perfil = perfil.loc[(partido["home_team"], 1)]
        except KeyError:
            local_perfil = None
        try:
            visita_perfil = perfil.loc[(partido["away_team"], 0)]
        except KeyError:
            visita_perfil = None

        # `dureza`, en cambio, sí es una Series (una sola métrica agrupada
        # por árbitro), así que `.get()` funciona correctamente aquí: busca
        # por el índice (nombre del árbitro) y devuelve None si no está.
        dureza_arbitro = dureza.get(partido["referee"])

        # Si algún equipo/árbitro no tiene historia previa suficiente (por
        # ejemplo, un equipo recién ascendido sin partidos en temporadas
        # anteriores, o un árbitro debutante), no hay forma honesta de darle
        # un feature pre-partido — se descarta esa fila en vez de rellenar
        # con un valor inventado que sesgaría el modelo.
        if local_perfil is None or visita_perfil is None or dureza_arbitro is None:
            continue

        filas.append({
            "season": temporada,
            "ganador": partido["ganador"],
            "total_game_cards": partido["total_game_cards"],
            "local_goles_favor": local_perfil["goals_for"],
            "local_goles_contra": local_perfil["goals_against"],
            "local_corners_favor": local_perfil["corners_for"],
            "local_tarjetas_favor": local_perfil["cards_for"],
            "visita_goles_favor": visita_perfil["goals_for"],
            "visita_goles_contra": visita_perfil["goals_against"],
            "visita_corners_favor": visita_perfil["corners_for"],
            "visita_tarjetas_favor": visita_perfil["cards_for"],
            "arbitro_dureza": dureza_arbitro,
        })

    return pd.DataFrame(filas)


features = construir_features(df)
print(f"{len(features)} partidos con features pre-partido completos, de {len(df)} totales")
print(f"(se pierden los {len(df) - len(features)} partidos de la primera temporada 2020-21, sin historia previa)")
features.head()

1636 partidos con features pre-partido completos, de 2102 totales
(se pierden los 466 partidos de la primera temporada 2020-21, sin historia previa)


,season,ganador,total_game_cards,local_goles_favor,local_goles_contra,local_corners_favor,local_tarjetas_favor,visita_goles_favor,visita_goles_contra,visita_corners_favor,visita_tarjetas_favor,arbitro_dureza
0,2021-2022,Empate,4,1.647059,1.000000,4.764706,1.705882,1.315789,1.368421,5.315789,1.842105,4.888889
1,2021-2022,Visitante,4,0.941176,1.352941,4.352941,2.352941,0.750000,1.200000,4.900000,1.850000,4.428571
2,2021-2022,Visitante,2,0.823529,1.058824,4.588235,2.470588,1.050000,1.950000,3.700000,2.700000,4.640000
3,2021-2022,Local,2,1.190476,1.047619,5.714286,2.095238,1.300000,1.200000,4.150000,2.300000,5.750000
4,2021-2022,Visitante,0,1.150000,1.000000,5.100000,1.150000,1.000000,2.058824,4.470588,2.588235,4.826087


In [4]:
# Split entrenamiento/validación: igual criterio que el notebook 15 -- la
# temporada COMPLETA más reciente (2025-26) como holdout genuino, nunca
# usada para ajustar ningún modelo de esta sección; el resto de temporadas
# con feature disponible (2021-22 a 2024-25) para entrenar. "2026-2027"
# (Apertura en curso, incompleta) se excluye de ambos conjuntos -- no sirve
# ni para entrenar con ella completa ni para validar contra un total real.
temporada_valid_modelo = "2025-2026"
entrenamiento_features = features[
    ~features["season"].isin([temporada_valid_modelo, "2026-2027"])
].copy()
validacion_features = features[features["season"] == temporada_valid_modelo].copy()

print(f"Entrenamiento: {len(entrenamiento_features)} partidos")
print(f"Validación ({temporada_valid_modelo}, holdout genuino): {len(validacion_features)} partidos")

Entrenamiento: 1278 partidos
Validación (2025-2026, holdout genuino): 312 partidos


## 4. Modelo A — Ganador (logística multinomial), solo con perfiles pre-partido

Mismo tipo de modelo que el notebook 11 (`sm.MNLogit`), pero con predictores 100% pre-partido:
los 4 perfiles del local + los 4 del visitante (8 variables en total). Comparamos el accuracy en
el holdout contra el baseline ingenuo de "predecir siempre Local" — el mismo criterio que usa el
notebook 11, para que los resultados sean comparables en espíritu aunque sean ligas distintas.

In [5]:
columnas_features_ganador = [
    "local_goles_favor", "local_goles_contra", "local_corners_favor", "local_tarjetas_favor",
    "visita_goles_favor", "visita_goles_contra", "visita_corners_favor", "visita_tarjetas_favor",
]

y_map = {"Empate": 0, "Local": 1, "Visitante": 2}

X_train = sm.add_constant(entrenamiento_features[columnas_features_ganador].astype(float))
y_train = entrenamiento_features["ganador"].map(y_map)

modelo_ganador = sm.MNLogit(y_train, X_train).fit(disp=False)
print(modelo_ganador.summary())

                          MNLogit Regression Results                          
Dep. Variable:                ganador   No. Observations:                 1278
Model:                        MNLogit   Df Residuals:                     1260
Method:                           MLE   Df Model:                           16
Date:                Wed, 09 Sep 2026   Pseudo R-squ.:                 0.03277
Time:                        21:59:34   Log-Likelihood:                -1325.0
converged:                       True   LL-Null:                       -1369.9
Covariance Type:            nonrobust   LLR p-value:                 2.735e-12
            ganador=1       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -1.5781      1.534     -1.029      0.304      -4.585       1.428
local_goles_favor         0.8572      0.294      2.912      0.004       0.280       1.434
local_go

In [6]:
# Evaluación en el holdout genuino (2025-26) — nunca visto por el modelo.
X_valid = sm.add_constant(
    validacion_features[columnas_features_ganador].astype(float), has_constant="add"
)
probabilidades = modelo_ganador.predict(X_valid)  # columnas en el orden 0,1,2 = Empate, Local, Visitante

y_map_inverso = {0: "Empate", 1: "Local", 2: "Visitante"}
prediccion = probabilidades.idxmax(axis=1).map(y_map_inverso)

accuracy = (prediccion.values == validacion_features["ganador"].values).mean()
baseline_local = (validacion_features["ganador"] == "Local").mean()

print(f"Accuracy del modelo (holdout 2025-26): {accuracy:.1%}")
print(f"Baseline ingenuo (predecir siempre 'Local'): {baseline_local:.1%}")
print(f"¿Mejora sobre el baseline?: {'Sí' if accuracy > baseline_local else 'No'} ({(accuracy - baseline_local)*100:+.1f} puntos porcentuales)")

Accuracy del modelo (holdout 2025-26): 49.4%
Baseline ingenuo (predecir siempre 'Local'): 46.8%
¿Mejora sobre el baseline?: Sí (+2.6 puntos porcentuales)


In [7]:
# Matriz de confusión: no solo importa el accuracy total, sino EN QUÉ se
# equivoca el modelo. Un modelo que solo predice "Local" siempre podría
# tener un accuracy parecido al baseline sin aportar nada real — la matriz
# de confusión deja ver si de verdad distingue entre las 3 clases.
matriz_confusion = pd.crosstab(
    validacion_features["ganador"], prediccion.values,
    rownames=["Real"], colnames=["Predicho"]
)
matriz_confusion

Predicho,Local,Visitante
Real,,
Empate,65,15
Local,137,9
Visitante,69,17


## 5. Modelo B — Total de tarjetas del partido, con features pre-partido

Los notebooks 12 y 14 ya establecieron, dentro de la misma muestra (ajuste, no predicción fuera
de muestra), que la identidad del árbitro explica más varianza de las tarjetas que la identidad
de los equipos. Aquí hacemos la pregunta complementaria y más exigente: **de verdad, prediciendo
un partido que el modelo nunca vio (2024-25), ¿ayuda más conocer el perfil de los equipos, la
dureza del árbitro, o ambos juntos?**

Ajustamos 3 versiones del mismo Poisson GLM sobre el mismo entrenamiento (2021-22 a 2023-24) y
las evaluamos las 3 contra el mismo holdout (2024-25), comparando MAE y correlación de la
predicción contra el total de tarjetas real:

- **Solo equipos**: `local_tarjetas_favor` + `visita_tarjetas_favor`.
- **Solo árbitro**: `arbitro_dureza`.
- **Completo**: ambos.

In [8]:
def ajustar_y_evaluar_cards(columnas_predictoras, nombre):
    """Ajusta un Poisson GLM sobre `entrenamiento_features` y evalúa su
    predicción fuera de muestra contra `validacion_features` (2025-26,
    nunca usado para ajustar el modelo).

    Se reporta MAE (en tarjetas, unidad fácil de interpretar) y
    correlación (qué tan bien ordena los partidos de más a menos
    tarjetero, aunque el nivel exacto no sea perfecto) -- los mismos dos
    criterios que se han usado en los notebooks 12 y 15 para poder comparar
    resultados de forma consistente en todo el proyecto.
    """
    X_train = sm.add_constant(entrenamiento_features[columnas_predictoras].astype(float))
    y_train = entrenamiento_features["total_game_cards"].astype(float)
    modelo = sm.GLM(y_train, X_train, family=sm.families.Poisson()).fit()

    X_valid = sm.add_constant(
        validacion_features[columnas_predictoras].astype(float), has_constant="add"
    )
    prediccion = modelo.predict(X_valid)
    real = validacion_features["total_game_cards"].astype(float)

    return {
        "modelo": nombre,
        "n_predictores": len(columnas_predictoras),
        "mae_holdout": (real - prediccion).abs().mean(),
        "correlacion_holdout": real.corr(prediccion),
    }


resultados_cards = pd.DataFrame([
    ajustar_y_evaluar_cards(["local_tarjetas_favor", "visita_tarjetas_favor"], "Solo equipos"),
    ajustar_y_evaluar_cards(["arbitro_dureza"], "Solo árbitro"),
    ajustar_y_evaluar_cards(["local_tarjetas_favor", "visita_tarjetas_favor", "arbitro_dureza"], "Completo (equipos + árbitro)"),
])
resultados_cards.round(3)

,modelo,n_predictores,mae_holdout,correlacion_holdout
0,Solo equipos,2,1.983,0.032
1,Solo árbitro,1,1.983,0.107
2,Completo (equipos + árbitro),3,1.983,0.098


## 6. Conclusión general del notebook

**Actualización (dataset extendido a 7 temporadas, holdout ahora es 2025-26 completa en vez de
2024-25): el resultado del Modelo A cambió de verdad, no es solo ruido de redondeo.**

**Modelo A (ganador):** con SOLO perfiles de equipo pre-partido, el modelo ahora **sí le gana al
baseline ingenuo** de "predecir siempre Local" — **49.4% de accuracy contra 46.8% del baseline**
(+2.6 puntos porcentuales, ver sección 4) — antes, con el dataset de 5 temporadas y holdout
2024-25, el modelo perdía contra el baseline (46.1% vs. 49.2%). Con casi el doble de partidos de
entrenamiento (1,278 vs. 957), el modelo encontró algo de señal real: identifica algunos partidos
donde el perfil del visitante es lo bastante fuerte frente al del local como para que valga la
pena predecir "Visitante" en vez de "Local" por default — eso es lo que explica la mejora (revisa
la matriz de confusión de la sección 4: sigue **sin predecir un solo empate** de los 80 reales en
el holdout, la ganancia viene enteramente de acertarle a más partidos de visitante).

Esto es una buena lección aparte: **con menos datos de entrenamiento, el modelo se veía peor de lo
que en realidad podía ser** — otro caso, van ya varios en este proyecto, de que una conclusión
puede cambiar con más evidencia. Sigue siendo un modelo modesto (nunca predice empates, el
pseudo-R² del ajuste es de ~3.3%), pero ya no es un caso claro de "no sirve para nada".

**Modelo B (tarjetas) — se mantiene el mismo patrón de antes:**

| Modelo | MAE (holdout) | Correlación (holdout) |
|---|---|---|
| Solo equipos | 1.983 | 0.032 |
| Solo árbitro | 1.983 | **0.107** |
| Completo (equipos + árbitro) | 1.983 | 0.098 |

Las correlaciones siguen siendo débiles, pero ahora todas positivas y el árbitro sigue siendo el
mejor predictor individual (0.107) — combinarlo con el equipo no ayuda (0.098, ligeramente peor).
**Esto no contradice los notebooks 12 y 14 — mide una pregunta distinta.** Esos notebooks usaban
dummies de CADA árbitro y CADA equipo dentro de la misma muestra de ajuste (eso es "explicar
varianza dentro de la muestra"). Aquí `arbitro_dureza` es un solo número (su promedio histórico)
usado para predecir un PARTIDO INDIVIDUAL nunca visto — y el total de tarjetas de un partido
concreto tiene mucha variación propia (desviación estándar dentro de un mismo árbitro ronda
2.0-2.9 tarjetas, ver notebook 14 sección 1), así que conocer el promedio del árbitro ayuda algo,
pero no mucho, para acertarle a un partido puntual.

**La lección metodológica de este notebook se sostiene igual de bien con más datos:** que una
variable "explique varianza" en un modelo con muchas categorías (fixed effects) no garantiza que
sirva para predecir una observación nueva con un resumen simple de esa variable — son dos
preguntas distintas ("¿se asocia X con Y en los datos que ya tengo?" vs. "¿puedo usar X para
adivinar Y en un caso nuevo?"), y aquí queda demostrado con números en ambas rondas de esta
actualización.

**Siguiente paso real:** con más datos el Modelo A mejoró solo, sin cambiar el método — vale la
pena reintentar este mismo notebook cuando la temporada 2026-27 (Apertura, en curso) termine y se
pueda usar como holdout más reciente. Para una mejora de fondo (no solo "esperar más datos") sigue
haciendo falta más contexto por partido: forma reciente de las últimas jornadas, historial cabeza
a cabeza entre los dos equipos específicos, o un sistema tipo ELO.